# 1. test 데이터 성능 비교

In [1]:
from pathlib import Path

import pandas as pd
from ultralytics import YOLO


ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_YAML = ROOT / "data" / "yolo_subset" / "data.yaml"

MODELS = {
    "baseline": 640,
    "improved_v1": 960,
    "improved_v2": 960,
}


rows = []

for name, img_size in MODELS.items():
    model = YOLO(str(ROOT / "models" / name / "weights" / "best.pt"))

    result = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=img_size,
        conf=0.25,
        verbose=False,
        plots=False,
    )

    p, r = result.box.mp, result.box.mr

    rows.append([
        name, img_size, p, r, 2 * p * r / (p + r),
        result.box.map50, result.box.map75, result.box.map,
        result.speed["inference"],
    ])


columns = [
    "Model", "Image Size", "Precision", "Recall", "F1",
    "mAP50", "mAP75", "mAP50-95", "Inference(ms)"
]

df = pd.DataFrame(rows, columns=columns)

display(df.round(4))

Ultralytics 8.4.138  Python-3.11.16 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 185.959.1 MB/s, size: 122.1 KB)
val: Scanning C:\Users\PMS\Desktop\MS\project\small-drone-detection\data\yolo_subset\labels\test.cache... 733 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 733/733  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 8.6it/s 5.4s<0.2s
                   all        733        749      0.948      0.884      0.914      0.614
Speed: 2.9ms preprocess, 2.5ms inference, 0.0ms loss, 0.2ms postprocess per image
Ultralytics 8.4.138  Python-3.11.16 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 609.2282.5 MB/s, size: 85.2 K

,Model,Image Size,Precision,Recall,F1,mAP50,mAP75,mAP50-95,Inference(ms)
0,baseline,640,0.9483,0.8838,0.9149,0.9143,0.7051,0.6139,2.4686
1,improved_v1,960,0.9661,0.9252,0.9452,0.9405,0.7500,0.6561,3.2162
2,improved_v2,960,0.9704,0.9372,0.9535,0.9419,0.7801,0.6852,2.9315


### Test 성능 비교

| Model | Image Size | Precision | Recall | F1 | mAP50 | mAP75 | mAP50-95 | Inference(ms) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Baseline | 640 | 0.9483 | 0.8838 | 0.9149 | 0.9143 | 0.7051 | 0.6139 | 2.47 |
| Improved V1 | 960 | 0.9661 | 0.9252 | 0.9452 | 0.9405 | 0.7500 | 0.6561 | 3.22 |
| Improved V2 | 960 | **0.9704** | **0.9372** | **0.9535** | **0.9419** | **0.7801** | **0.6852** | 2.93 |

### 최종 판단

- Baseline 대비 V1은 입력 크기를 640에서 960으로 확대하면서 Recall이 0.8838 → 0.9252, mAP50-95가 0.6139 → 0.6561로 개선됨.
- V2는 V1 대비 Recall이 0.0120, mAP75가 0.0301, mAP50-95가 0.0291 추가 상승하여 작은 객체 탐지와 위치 정확도 개선이 최종 Test에서도 유지됨.
- Baseline 대비 최종 V2는 Precision +0.0221, Recall +0.0534, F1 +0.0386, mAP50-95 +0.0713의 개선을 확인함.
- V2의 추론 시간은 2.93ms로 Baseline보다 증가했지만, 동일한 960 입력을 사용하는 V1의 3.22ms보다 빠르며 성능도 모든 주요 지표에서 가장 높음.
- 따라서 **Improved V2를 최종 모델로 선정**함.